<a href="https://colab.research.google.com/github/Learn-AIMLOps/Instruction_fine_tunning/blob/main/push_IFT_dataset_HF.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [ ]:
# Importing Google Colab Secrets
from google.colab import userdata

In [ ]:
# Load HuggingFace Token (Colab Secret)
hf_write_token = userdata.get("HF_WRITE_TOKEN")

In [ ]:
# Huggingface Login
from huggingface_hub import login
login(token=hf_write_token)

In [ ]:
!pip install -q datasets huggingface_hub

In [ ]:
# Define the cleaning function + load your file

import json
from datasets import Dataset

def load_robust_jsonl(file_path):
    data = []
    skipped = 0
    with open(file_path, 'r', encoding='utf-8') as f:
        for line_number, line in enumerate(f, start=1):
            try:
                entry = json.loads(line.strip())  # strip() helps with trailing whitespace
                # Extract and clean fields
                instruction = entry.get("instruction", "").strip()
                input_text   = entry.get("input", "").strip()
                response     = entry.get("response", "")

                # Skip if critical fields are empty
                if not instruction or not response:
                    skipped += 1
                    continue

                # Handle non-string responses (dict, list, etc.)
                if not isinstance(response, str):
                    response = json.dumps(response, ensure_ascii=False, indent=None)

                # Final cleaned entry
                cleaned = {
                    "instruction": instruction,
                    "input": input_text,
                    "response": response.strip()
                }
                data.append(cleaned)
            except json.JSONDecodeError:
                skipped += 1
                continue
            except Exception as e:
                print(f"Unexpected error at line {line_number}: {e}")
                skipped += 1
                continue

    print(f"Loaded {len(data)} valid samples.")
    print(f"Skipped {skipped} malformed / empty samples.")

    if not data:
        raise ValueError("No valid entries found in the file.")

    return Dataset.from_list(data)

In [ ]:
# Load & clean your JSONL file
# If the file is already uploaded to Colab
file_path = "/content/ift_dataset.jsonl"          # ← CHANGE THIS

# If it's in Google Drive (uncomment if needed)
# from google.colab import drive
# drive.mount('/content/drive')
# file_path = "/content/drive/MyDrive/your_dataset.jsonl"

clean_dataset = load_robust_jsonl(file_path)

# Quick look
print(clean_dataset)
print(clean_dataset[0])               # first example
print(f"Total examples after cleaning: {len(clean_dataset)}")

Loaded 91 valid samples.
Skipped 9 malformed / empty samples.
Dataset({
    features: ['instruction', 'input', 'response'],
    num_rows: 91
})
{'instruction': 'Explain why small language models offer a more resource-efficient alternative than larger ones.', 'input': 'Small language models are often used in conjunction with large-scale pre-trained models to scale up the computational resources required to train them.', 'response': 'Small language models require less computational power and data to train compared to larger models, making them a more resource-efficient option.'}
Total examples after cleaning: 91


In [ ]:
# Push the cleaned dataset to the Hub
# Choose your repo name (username must match your HF account)
repo_id = "RohitGu/llmops-guide-ift-datasets"

# Push (creates repo if it doesn't exist)
clean_dataset.push_to_hub(
    repo_id,
    private=False,           # set True if you want it private
    # commit_message="Initial cleaned upload of instruction dataset"
    token=hf_write_token
)

print(f"Dataset pushed to: https://huggingface.co/datasets/{repo_id}")

Uploading the dataset shards:   0%|          | 0/1 [00:00<?, ? shards/s]

Creating parquet from Arrow format:   0%|          | 0/1 [00:00<?, ?ba/s]

Processing Files (0 / 0)      : |          |  0.00B /  0.00B            

New Data Upload               : |          |  0.00B /  0.00B            

                              : 100%|##########| 30.0kB / 30.0kB            

                              : 100%|##########| 30.0kB / 30.0kB            

Dataset pushed to: https://huggingface.co/datasets/RohitGu/llmops-guide-ift-datasets
